# 🚀 PLAN 1: PERFORMANCE — NVIDIA A100 (40GB VRAM Dual-Resident • ~5.38 CU/h)

> **Mục đích:** Chế độ hiệu năng cao nhất trên GPU A100 40GB (~5.38 CU/h, CU > 35) cho bài toán kiến trúc phức tạp, refactor diện rộng hoặc phân tích chuyên sâu.
> **Dual-Resident (`OLLAMA_MAX_LOADED_MODELS="2"`):** Thường trú đồng thời cả Conductor (`deepseek-r1:14b`) và Worker (`qwen2.5-coder:32b` 🏆 / `qwen3-coder:30b`) trong 40GB VRAM với độ trễ chuyển đổi 0.0s.

---

### 🧩 Phân Bổ Mô Hình (Dual-Resident Protocol)

| Vai trò | Mô hình | VRAM | Nhiệm vụ chính |
| :--- | :--- | :--- | :--- |
| 🎼 **Conductor** | **`deepseek-r1:14b`** | ~9.0 GB | Phân tích yêu cầu, tư duy `<think>`, xuất checklist `.agents/kb/HANDOFF.md` |
| 🛠️ **Worker** | **`qwen2.5-coder:32b` 🏆**<br>*(Hoặc `qwen3-coder:30b` MoE)* | ~19.8 GB | Thực thi chuỗi task (Code, Unit Test, Workflow CLI/JSON, Mermaid Diagram) |

---

### 🛡️ Cấu Hình Dual-Resident VRAM (A100 40GB)
- **Thiết lập:** `OLLAMA_MAX_LOADED_MODELS="2"`, `OLLAMA_FLASH_ATTENTION="1"`, `OLLAMA_KV_CACHE_TYPE="q8_0"`.
- **Đỉnh VRAM (32K context):** `34.5 GB / 40.0 GB` (86% VRAM: 9.0 GB M1 + 19.8 GB M2 + 5.7 GB KV Cache).
- **Độ trễ chuyển đổi (0.0s):** Cả hai mô hình nằm cố định trong VRAM suốt phiên, không phải reload và không mất KV Cache.

---

💡 **Notebook liên quan:**
- 🏆 **T4 16GB Tiết kiệm CU (~1.53 CU/h):** [`colab_ollama_plan1_performance_t4.ipynb`](colab_ollama_plan1_performance_t4.ipynb)
- ⚡ **T4 Dual-Resident Fast (CU < 35 hoặc 65–75 tok/s):** [`colab_ollama_plan2_fast_t4.ipynb`](colab_ollama_plan2_fast_t4.ipynb)
- 📘 **Bản Tutorial cơ bản (1-Model):** [`colab_ollama_tutorial.ipynb`](colab_ollama_tutorial.ipynb)

### Step 1: Xác Thực Phần Cứng NVIDIA A100 (40GB VRAM)
Kiểm tra máy ảo Google Colab đã được cấp phát GPU **NVIDIA A100-SXM4-40GB**.

In [ ]:
# Step 1: Kiểm tra phần cứng NVIDIA GPU và dung lượng VRAM thực tế
!nvidia-smi

### Step 2: Cấu Hình Lưu Trữ Mô Hình (Colab NVMe SSD hoặc Google Drive)
Thiết lập nơi lưu trữ trọng số mô hình:
- **`USE_GDRIVE = False` (Khuyên dùng):** Lưu trên Colab NVMe SSD nội bộ (`/root/.ollama/models`) cho tốc độ cao và không cần cấp quyền Google Drive.
- **`USE_GDRIVE = True`:** Lưu trên Google Drive nếu cần duy trì cache qua nhiều phiên làm việc trên trình duyệt.

In [ ]:
import os

# Đặt True nếu chạy trên trình duyệt (Safari/Chrome) và muốn lưu model cố định vào Google Drive
# Mặc định False để ưu tiên tốc độ NVMe SSD nội bộ (>600 MB/s) & tương thích 100% với IDE Kernel
USE_GDRIVE = False

if USE_GDRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        models_dir = '/content/drive/MyDrive/ollama_models'
        os.environ['OLLAMA_MODELS'] = models_dir
        os.makedirs(models_dir, exist_ok=True)
        print(f"✅ Đã kích hoạt Google Drive cache tại: {models_dir}")
    except Exception as e:
        print(f"ℹ️ Đang chạy trong IDE Kernel (Bỏ qua Google Drive mount: {e}).")
        print("⚡ Sử dụng ổ cứng Colab NVMe SSD nội bộ (/root/.ollama/models).")
else:
    print("⚡ Sử dụng ổ cứng Colab NVMe SSD nội bộ (/root/.ollama/models) — Tối ưu tốc độ nạp model.")

### Step 3: Cài Đặt Ollama & Khởi Chạy Daemon Dual-Resident
Cài đặt Ollama, cấu hình Swapfile và kích hoạt daemon chế độ Dual-Resident:
- **16GB NVMe Swapfile:** Tạo vùng nhớ hoán đổi bảo vệ hệ thống.
- **Dual-Resident (`OLLAMA_MAX_LOADED_MODELS="2"`):** Cho phép thường trú đồng thời cả Conductor và Worker trong VRAM.
- **Tối ưu VRAM:** Kích hoạt `OLLAMA_FLASH_ATTENTION="1"` và `OLLAMA_KV_CACHE_TYPE="q8_0"`, kiểm soát tổng mức dùng ở mức `34.5 GB / 40 GB` (86%) ở context 32K.

In [ ]:
import os
import subprocess
import time
import urllib.request
from IPython.display import clear_output

# 1. Kích hoạt 16GB Linux NVMe Swapfile
!test -f /swapfile || (fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile)

# 2. Cài đặt zstd và bản phát hành Ollama Linux chính thức
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Cấu hình biến môi trường Dual-Resident cho A100 40GB VRAM
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "24h"          # Giữ cố định cả 2 model trong VRAM 24h
env["OLLAMA_MAX_LOADED_MODELS"] = "2"      # DUAL-RESIDENT: Ghim đồng thời M1 (9GB) + M2 (19.8GB) trong 40GB VRAM!
env["OLLAMA_NUM_PARALLEL"] = "2"          # Xử lý song song mượt mà trong trần 34.5GB / 40GB VRAM
env["OLLAMA_FLASH_ATTENTION"] = "1"       # Bật Flash Attention tăng tốc độ prefill & decode
env["OLLAMA_KV_CACHE_TYPE"] = "q8_0"      # Nén KV Cache 8-bit, giữ KV 32K ctx chỉ ~5.7 GB VRAM

# 4. Khởi chạy 'ollama serve' ngầm
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
clear_output(wait=True)
time.sleep(3)

try:
    with urllib.request.urlopen("http://localhost:11434/") as resp:
        print("🎉 Plan 1 A100 Dual-Resident Daemon sẵn sàng (MAX_LOADED_MODELS=2 | FLASH_ATTENTION=1 | KV=q8_0):")
        print(resp.read().decode("utf-8"))
except Exception as e:
    print("⚠️ Không thể kết nối Ollama daemon:", e)

### Step 4: Tải & Ghim Đồng Thời Cả 2 Mô Hình Trong 40GB VRAM
Tải mô hình và kích hoạt thường trú đồng thời:
- **Mặc định (`USE_QWEN3_MOE = False`):** Sử dụng **`qwen2.5-coder:32b` 🏆** (~19.8 GB) cho độ chính xác logic và kiến trúc cao nhất.
- **Chế độ tốc độ cao (`USE_QWEN3_MOE = True`):** Đổi 1 dòng sang **`qwen3-coder:30b`** (MoE 3.3B active params, tốc độ ~130 tok/s).
- Tự động nạp đồng thời cả **`deepseek-r1:14b`** và mô hình Worker vào VRAM A100 (độ trễ chuyển đổi 0.0s).

In [ ]:
import json
import urllib.request
from IPython.display import clear_output

# ==============================================================================
# 🎛️ CÔNG TẮC 1 DÒNG CHỌN MODEL 2 (CHUYÊN GIA CODING TRÊN A100):
# - False (Mặc định 🏆): Dùng "qwen2.5-coder:32b" (Dense 32.5B — chất lượng code & refactor sâu nhất)
# - True  (Siêu tốc ⚡): Dùng "qwen3-coder:30b"   (MoE 30B-A3B — tốc độ sinh code ~130 tok/s)
# ==============================================================================
USE_QWEN3_MOE = False

CONDUCTOR_MODEL = "deepseek-r1:14b"
WORKER_MODEL = "qwen3-coder:30b" if USE_QWEN3_MOE else "qwen2.5-coder:32b"

A100_DUAL_MODELS = [
    (CONDUCTOR_MODEL, "🎼 Model 1 (Nhạc Trưởng): DeepSeek CoT <think> Planning & Reasoning (~9.0 GB)"),
    (WORKER_MODEL,    "🛠️ Model 2 (Chuyên Gia 🏆): Flagship Coding, Workflow & Diagram (~19.8 GB)"),
]

print("⏳ Đang tải bộ đôi Dual-Resident cho A100 40GB...")
for tag, role in A100_DUAL_MODELS:
    print(f"\n📦 Đang tải {tag} — {role}...")
    !ollama pull {tag}

print("\n🔥 Đang ghim đồng thời cả 2 model vào 40GB GPU VRAM (Dual-Resident 0.0s Switch)...")
for tag, _ in A100_DUAL_MODELS:
    try:
        payload = json.dumps({"model": tag, "prompt": "hi", "stream": False, "keep_alive": "24h"}).encode("utf-8")
        req = urllib.request.Request("http://localhost:11434/api/generate", data=payload, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=180):
            print(f"  ✨ Đã ghim cố định vào VRAM: {tag}")
    except Exception as e:
        print(f"  ⚠️ Lưu ý khi nạp {tag}: {e}")

clear_output(wait=True)
print(f"✅ DUAL-RESIDENT HOÀN TẤT! Cả {CONDUCTOR_MODEL} và {WORKER_MODEL} đang nằm chung trong 40GB VRAM (0.0s switch)!")
!ollama list

### Step 5: Mở Cloudflare Quick Tunnel & Kết Nối Với IDE
Khởi tạo đường hầm HTTPS từ `localhost:11434` ra URL công khai `*.trycloudflare.com`:
- **URL Tunnel:** Tự động lấy URL để cấu hình vào MCP Client hoặc IDE.
- **Xả buffer (Pipe Drainer):** Chạy luồng nền tự động xả `stdout` của `cloudflared`, ngăn ngừa tràn pipe buffer 64KB và lỗi treo đường hầm (HTTP 524).

In [ ]:
import re
import subprocess
import threading
import time
from IPython.display import clear_output

# 1. Tải và cài đặt Cloudflare Tunnel (cloudflared)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Khởi chạy cloudflared chuyển tiếp HTTPS về localhost:11434
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Đọc stdout để trích xuất URL '*.trycloudflare.com'
tunnel_url = None
print("⏳ Đang thiết lập Cloudflare Quick Tunnel...")
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.2)

# 4. QUAN TRỌNG: Chạy thread nền xả stdout liên tục để tránh đầy bộ đệm OS Pipe 64KB gây treo tunnel
def _drain_tunnel_logs(proc):
    try:
        for _ in iter(proc.stdout.readline, ""):
            pass
    except Exception:
        pass

threading.Thread(target=_drain_tunnel_logs, args=(tunnel_proc,), daemon=True).start()

clear_output(wait=True)

if tunnel_url:
    print("=" * 78)
    print("🎉 PLAN 1: PERFORMANCE (A100 40GB DUAL-RESIDENT) TUNNEL IS ONLINE!")
    print(f"🔗 PUBLIC ENDPOINT URL: {tunnel_url}")
    print("=" * 78)
    print("\n🚀 CÁCH 1 (Dán trực tiếp URL vào khung chat IDE / CLI):\n")
    print(f"   {tunnel_url}")
    print("   (Agent sẽ tự động gọi `ollama_set_endpoint`, kiểm tra kết nối và chuyển sang GPU Colab!)\n")
    print("⚡ CÁCH 2 (Chạy lệnh 1 dòng trong Terminal tại thư mục dự án):\n")
    print(f"   python3 colab_ollama_mcp/server.py --set-endpoint {tunnel_url}")
    print("\n" + "-" * 78)
else:
    print("❌ Không lấy được Cloudflare Tunnel URL. Vui lòng chạy lại cell này.")

### Step 6: Duy Trì Kết Nối Phiên Dài (Keep-Alive)
Kích hoạt phần tử HTML5 Silent Audio giúp tab trình duyệt duy trì trạng thái *Media Active*, ngăn chặn Colab bị ngắt kết nối hoặc đóng băng khi chạy nền.

In [ ]:
from IPython.display import HTML, display

display(HTML('''
<div style="padding: 14px; background: #e8f5e9; border: 1px solid #4caf50; border-radius: 8px; font-family: sans-serif;">
  <b style="color: #2e7d32;">🟢 Keep-Alive Active (Silent Audio Loop)</b>
  <p style="margin: 4px 0 8px 0; color: #1b5e20; font-size: 13px;">
     Giữ phiên Google Colab luôn hoạt động ổn định trong suốt quá trình bạn code trên IDE.
  </p>
  <audio loop autoplay controls style="height: 30px;">
    <source src="data:audio/wav;base64,UklGRsQAAABXQVZFZm10IBAAAAABAAEAQB8AAIA+AAACABAAZGF0YaAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA" type="audio/wav">
  </audio>
</div>
'''))

### Step 7: Bảng Theo Dõi Tài Nguyên Thời Gian Thực (Telemetry)
Theo dõi trạng thái GPU VRAM, System RAM, Swap, Disk và các mô hình đang nạp trong VRAM (`/api/ps`):
- **`AUTO_REFRESH = False` (Mặc định):** Hiển thị nhanh snapshot tài nguyên hiện tại.
- **`AUTO_REFRESH = True`:** Bật dashboard tự động làm mới mỗi 5 giây (`clear_output(wait=True)`).

In [ ]:
import json
import shutil
import subprocess
import time
import urllib.request
from IPython.display import clear_output

AUTO_REFRESH = False
REFRESH_SECONDS = 5

def make_bar(pct: float, width: int = 24) -> str:
    filled = max(0, min(width, int(round((pct / 100.0) * width))))
    return "█" * filled + "░" * (width - filled)

def print_telemetry() -> None:
    print("=" * 78)
    print(f"📊 PLAN 1 PERFORMANCE (A100 DUAL-RESIDENT) TELEMETRY — {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
    print("=" * 78)

    # 1. GPU VRAM, Load & Temperature
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,temperature.gpu,utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ], text=True).strip()
        for line in out.splitlines():
            name, temp, util, mem_used, mem_total = [x.strip() for x in line.split(",")]
            used_gb = float(mem_used) / 1024.0
            total_gb = float(mem_total) / 1024.0
            vram_pct = (float(mem_used) / float(mem_total)) * 100.0 if float(mem_total) > 0 else 0.0
            util_pct = float(util)
            print(f"🎮 GPU Hardware : {name}  (🌡️ {temp}°C)")
            print(f"⚡ GPU Core Load: [{make_bar(util_pct)}] {util_pct:5.1f}%")
            print(f"🧠 GPU VRAM     : [{make_bar(vram_pct)}] {vram_pct:5.1f}%  ({used_gb:.2f} GB / {total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ GPU Status   : {e}")

    # 2. System RAM, Page Cache & Swapfile
    try:
        meminfo = {}
        with open("/proc/meminfo", "r") as f:
            for line in f:
                parts = line.split(":")
                if len(parts) == 2:
                    meminfo[parts[0].strip()] = int(parts[1].strip().split()[0])
        total_ram_gb = meminfo.get("MemTotal", 0) / (1024 ** 2)
        avail_ram_gb = meminfo.get("MemAvailable", 0) / (1024 ** 2)
        used_ram_gb = total_ram_gb - avail_ram_gb
        ram_pct = (used_ram_gb / total_ram_gb * 100.0) if total_ram_gb > 0 else 0.0
        cached_gb = meminfo.get("Cached", 0) / (1024 ** 2)
        swap_total_gb = meminfo.get("SwapTotal", 0) / (1024 ** 2)
        swap_free_gb = meminfo.get("SwapFree", 0) / (1024 ** 2)
        swap_used_gb = swap_total_gb - swap_free_gb
        print(f"💻 System RAM   : [{make_bar(ram_pct)}] {ram_pct:5.1f}%  ({used_ram_gb:.2f} GB / {total_ram_gb:.2f} GB | Page Cache: {cached_gb:.2f} GB)")
        print(f"🛡️ NVMe Swapfile: {swap_used_gb:.2f} GB / {swap_total_gb:.2f} GB (Tổng bộ nhớ ảo: {total_ram_gb + swap_total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ System RAM   : {e}")

    # 3. Colab SSD Disk Usage
    try:
        total_d, used_d, _ = shutil.disk_usage("/")
        used_d_gb = used_d / (1024 ** 3)
        total_d_gb = total_d / (1024 ** 3)
        disk_pct = (used_d / total_d * 100.0) if total_d > 0 else 0.0
        print(f"💾 Colab Disk   : [{make_bar(disk_pct)}] {disk_pct:5.1f}%  ({used_d_gb:.1f} GB / {total_d_gb:.1f} GB)")
    except Exception as e:
        print(f"⚠️ Colab Disk   : {e}")

    # 4. Active Models Pinned in GPU VRAM (/api/ps)
    print("-" * 78)
    try:
        with urllib.request.urlopen("http://localhost:11434/api/ps", timeout=5) as resp:
            ps_data = json.loads(resp.read().decode("utf-8"))
            running = ps_data.get("models", [])
            if running:
                print(f"🔥 Đang ghim trong GPU VRAM ({len(running)} model):")
                for m in running:
                    m_name = m.get("name", "unknown")
                    vram_gb = m.get("size_vram", 0) / (1024 ** 3)
                    total_gb = m.get("size", 0) / (1024 ** 3)
                    gpu_offload = (vram_gb / total_gb * 100.0) if total_gb > 0 else 100.0
                    print(f"   • {m_name} — VRAM: {vram_gb:.2f} GB ({gpu_offload:.0f}% GPU offload)")
            else:
                print("💤 GPU VRAM hiện đang rảnh (Model sẽ nạp tức thì khi có request đầu tiên)")
    except Exception as e:
        print(f"🔴 Ollama Status: Offline hoặc không phản hồi ({e})")
    print("=" * 78)

try:
    while True:
        clear_output(wait=True)
        print_telemetry()
        if not AUTO_REFRESH:
            break
        print(f"🔄 Tự động làm mới mỗi {REFRESH_SECONDS}s... (Nhấn Stop ⏹️ để dừng)")
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("⏹️ Đã dừng bảng theo dõi tài nguyên.")